# Task 8: BERT Finetuning - Classification

Finetuning BERT for sentiment analysis.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

In [ ]:
class BERTForClassification(nn.Module):
    """BERT for sequence classification."""
    
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, 
                 d_ff=3072, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = BERTEncoder(vocab_size, d_model, num_heads, num_layers, d_ff, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        
    def forward(self, token_ids, attention_mask=None, labels=None):
        _, pooled = self.bert(token_ids, attention_mask=attention_mask)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        
        return loss, logits


# Simplified encoder for demo
class BERTEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, d_ff=3072, dropout=0.1):
        super().__init__()
        self.embeddings = BERTEmbeddings(vocab_size, d_model)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.pooler = nn.Linear(d_model, d_model)
        
    def forward(self, token_ids, attention_mask=None):
        x = self.embeddings(token_ids)
        for layer in self.encoder_layers:
            x = layer(x)
        pooled = torch.tanh(self.pooler(x[:, 0]))
        return x, pooled


class BERTEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.token = nn.Embedding(vocab_size, d_model)
        self.segment = nn.Embedding(2, d_model)
        self.position = nn.Embedding(512, d_model)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, token_ids):
        seq_len = token_ids.size(1)
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand_as(token_ids)
        return self.norm(self.token(token_ids) + self.position(positions))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, d_ff), nn.GELU(), nn.Dropout(dropout), 
                                nn.Linear(d_ff, embed_dim), nn.Dropout(dropout))
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.ffn(x))
        return x

print("BERT Classification model defined!")

In [ ]:
# Synthetic dataset for demo
class SentimentDataset(Dataset):
    def __init__(self, num_samples=1000, max_len=128, vocab_size=30000):
        self.token_ids = torch.randint(4, vocab_size, (num_samples, max_len))
        self.labels = torch.randint(0, 2, (num_samples,))  # 0=negative, 1=positive
        self.attention_mask = torch.ones(num_samples, max_len)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.token_ids[idx], self.attention_mask[idx], self.labels[idx]

# Create model (smaller for demo)
model = BERTForClassification(vocab_size=30000, d_model=128, num_heads=4, num_layers=2, 
                               d_ff=512, num_labels=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Training loop
dataset = SentimentDataset(num_samples=500)
dataloader = DataLoader(dataset, batch_size=16)

model.train()
for epoch in range(3):
    total_loss = 0
    for batch in dataloader:
        token_ids, attention_mask, labels = batch
        
        loss, _ = model(token_ids, attention_mask, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

print("\n✓ Training complete!")

In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for token_ids, attention_mask, labels in dataloader:
        _, logits = model(token_ids, attention_mask)
        predictions = logits.argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {correct/total*100:.2f}%")

## Summary
- ✅ Use [CLS] token for classification
- ✅ Finetune entire model on downstream task
- ✅ Cross-entropy loss for classification

## Next: [Task 9: BERT Finetuning - QA](../task-09-bert-qa/overview.html)